In [1]:
import pandas as pd
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from pathlib import Path
from typing import Literal
import pymupdf
import os
import re

In [2]:
FOLDER_PATH = "/Users/tomthomas/Documents/rag-foundation/data/papers"
paths = list(Path(FOLDER_PATH).glob("*.pdf"))

In [3]:
class Chunk(BaseModel):
    text: str
    chunk_id: str
    section_tag: Literal["References", "Content", "Appendix"]
    page_number: int = Field(ge=1)
    doc_id: str

class RetrievedChunk(BaseModel):
    chunk: Chunk
    score: float

class Citation(BaseModel):
    page_number: int = Field(ge=1)
    doc_id: str

class AnswerSentence(BaseModel):
    text: str
    citations: list[Citation]

class Answer(BaseModel):
    sentences: list[AnswerSentence]

In [4]:
def is_appendix_heading(lines: list[str], i: int) -> bool:
    s = lines[i].strip()
    if s.lower() == "appendix":
        return True
    if s == "A":
        for nxt in lines[i + 1:]:
            nxt = nxt.strip()
            if nxt:                      # first non-empty line after the "A"
                return nxt[0].isupper()
    return False

In [5]:
def load_pages(path: str) -> list[str]:
    """Returns list of pages from the given file path.
    
    Args:
        path (str): The file path to each paper.
    
    The page number of item i is i + 1 (pages are numbered from 1, matching the PDF page index).
    Returns:
        list: A list of pages from the given file path.
    
    """
    doc = pymupdf.open(path)
    pages = []
    for page in doc:
        pages.append(page.get_text())
    doc.close()
    return pages


In [6]:
def split_sections(pages: list[str]) -> dict[str, list[tuple[int, str]]]:
    """Returns a dictionary containing section and section pages.
        
        Args:
            pages (list[str]): list of pages from the document.

        Definition:
            A section is defined as a part of the document that starts with a section heading and ends before the next section heading.
            If no heading, then the section is by default content for the text.
            If reference is in middle of page, text after the reference is considered part of the reference section.
        
        Returns:
            dict: A dictionary containing section and section pages, where the key is the section name and the value is a list of tuples containing the page number and the text on that page.
        
        """
    sections = {
        "Content": [],
        "References": [],
        "Appendix": []
    }
    ref_found = False
    appendix_found = False
    for page_num, page in enumerate(pages, start=1):
        ref_text = ""
        appendix_text = ""
        content = ""
        lines = page.split("\n")
        for i, line in enumerate(lines):
            if ref_found and not is_appendix_heading(lines, i):
                ref_text += line + "\n"
            elif appendix_found and line.strip().lower() != "references":
                appendix_text += line + "\n"
            elif line.strip().lower() == "references":
                if appendix_found:
                    appendix_found = False
                ref_found = True
                ref_text += line + "\n"
            elif is_appendix_heading(lines, i) and ref_found:
                if ref_found:
                    ref_found = False
                appendix_found = True
                appendix_text += line + "\n"
            else:
                content += line + "\n"
        if content != "":
            sections["Content"].append((page_num, content))
        if ref_text != "":
            sections["References"].append((page_num, ref_text))
        if appendix_text != "":
            sections["Appendix"].append((page_num, appendix_text))
    return sections

In [84]:
def clean_section(section: list[tuple[int, str]]) -> list[tuple[int, list[str]]]:
    """Cleans a section by removing any empty text entries/removing unwanted new line strings between sentences 
        or removing emtpy lines or removing unwanted data (for eg: lines with only digits or removing roman letters etc.).

        Args:
            section (list[tuple[int, str]]): A list of tuples containing the page number and the text on that page.

        Returns:
            list[tuple[int, list[str]]]: A cleaned list of tuples containing the page number and a list of paragraphs on that page.

    """

    cleaned_section = []
    roman_numeral = re.compile(
        r"(?i)M{0,3}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})"
    )
    for page_number, text in section:
        sentences = []
        page_text = ""
        text = text.split("arXiv Template\nA PREPRINT\n")[-1]
        lines = text.split("\n")
        non_empty = [i for i, l in enumerate(lines) if l.strip()]
        if non_empty:
            first, last = non_empty[0], non_empty[-1]
        else:
            first, last = 0, len(lines) - 1
        for i, line in enumerate(lines):
            line = line.strip()
            if i == first and (line.isdigit() or line.startswith("Published")):
                continue
            if i == last and line.isdigit():
                continue
            page_text += line + "\n"
        paragraph = re.split(r"(?<=\.)\n", page_text)
        for sentence in paragraph:
            sentence = sentence.strip()
            sentence = sentence.replace("-\n", "")
            idx = sentence.find("\n")
            if idx != -1:
                sentence = sentence.replace("\n", " ")
            if (sentence.isdigit() or sentence.isspace() or not sentence.strip() or sentence.startswith(("arXiv","Preprint","URL","org","doi")) or "arXiv" in sentence or roman_numeral.fullmatch(sentence) or (sentence.isalpha() and len(sentence) ==1)) and (len(sentence) < 50):
                continue
            sentences.append(sentence)
        if sentences:
            cleaned_section.append((page_number, sentences))
    return cleaned_section


In [85]:
for path in paths:
    doc_id = path.stem
    pages = load_pages(path)
    # print(f"Processing document: {doc_id}")
    # print(f"Loaded {len(pages)} pages for document: {doc_id}")
    sections = split_sections(pages)
    # print(f"Split document: {doc_id} into sections: {list(sections.keys())}")
    for section_name, section_pages in sections.items():
        # print(f"Section '{section_name}' starts at page {section_pages} and has {len(section_pages)} pages.")
        cleaned_section = clean_section(section_pages)
        # print(f"Cleaned section '{section_name}' has {len(cleaned_section)} pages.")
        # for page_number, sentences in cleaned_section:
        #     # print(sentences)
        #     # print(f"Page {page_number} has {len(sentences)} sentences.")

In [86]:
cleaned_section

[(15,
  ['A SCALABILITY AND COMPUTATIONAL EFFICIENCY OF THE TREE-BUILDING PROCESS To assess the computational efficiency and cost-effectiveness of RAPTOR’s tree-building process, we conducted experiments on a consumer-grade laptop, specifically an Apple M1 Mac with 16GB of RAM. These experiments aimed to demonstrate the scalability and feasibility of RAPTOR on typical hardware. We varied the context length from 12,500 to 78,000 tokens and measured both the token expenditure and the time required to complete the tree-building process, from initial splitting and embedding to the construction of the final root node.',
   'Figure 5: Token cost as a function of document length for QASPER, NarrativeQA, and QuALITY.',
   'RAPTOR tree construction costs scale linearly with document length for each of the datasets.',
   'Token Expenditure We empirically investigated the relationship between the initial document length and the total number of tokens expended during the tree-building process, whi